In [1]:
import geopandas as gpd
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
# Load the shapefile containing the coastal profiles
profiles = gpd.read_file(
    os.path.join("data", "profiles",  "perfiles_playas_limpios_NoDuplicate.gpkg"),
    layer="perfiles_playas_limpios_NoDuplicate"
    )
# Ensure the "ID_Perfil" column is of integer type
profiles["ID_Perfil"] = profiles["ID_Perfil"].astype(int)
# Sort the profiles by "ID_Perfil" for better organization
profiles = profiles.sort_values(by="ID_Perfil").reset_index(drop=True)
# Recalculate the "ID_Perfil" column to ensure it is sequential and unique
profiles["ID_Perfil"] = range(1, len(profiles) + 1)

# Get a list of profile IDs whose 'ID_playa' is empty or NaN
empty_playa_ids = profiles[profiles['ID_playa'] == '']['ID_Perfil'].tolist()
empty_playa_rows = profiles[profiles['ID_Perfil'].isin(empty_playa_ids)]

# Remove profiles with empty 'ID_playa' values from the GeoDataFrame and change its data type to integer
profiles = profiles[profiles['ID_playa'] != ''].reset_index(drop=True)
profiles["ID_playa"] = profiles["ID_playa"].astype(int)

# Load the shorelines
shorelines = gpd.read_file(os.path.join("data", "shorelines",  "LineasCosta.gpkg"),
                           layer="LineasCostaPlayas")
# Keep only the relevant columns
columns_to_keep = ["ID_LineaCosta", "Mision", "Municipio", "Fecha", "Hora", "Cuadrante", "NivelTotal"]
#shorelines = shorelines[columns_to_keep + ["Playa_shoreline", "geometry"]]

# Ensure both GeoDataFrames use the same coordinate reference system (CRS)
profiles = profiles.to_crs(shorelines.crs)
print(f"Profiles CRS: {profiles.crs}")

Profiles CRS: EPSG:25830


In [3]:
# Intersect profiles and shorelines, retaining only point intersections.
profiles_shoreline_intersections = gpd.overlay(
    profiles[["ID_Perfil", "geometry"]],
    shorelines,
    how="intersection",
    keep_geom_type=False,
).explode(index_parts=False, ignore_index=True)
profiles_shoreline_intersections = profiles_shoreline_intersections[
    profiles_shoreline_intersections.geometry.geom_type == "Point"
].copy()

# Assign the authoritative beach from profiles immediately after the intersection.
perfil_playa = profiles[["ID_Perfil", "ID_playa"]].drop_duplicates(
    subset=["ID_Perfil"],
    keep="first",
)
assert perfil_playa["ID_Perfil"].is_unique
profiles_shoreline_intersections = profiles_shoreline_intersections.merge(
    perfil_playa,
    how="left",
    on="ID_Perfil",
    validate="many_to_one",
)

# Remove duplicate intersections before any downstream processing.
profiles_shoreline_intersections = profiles_shoreline_intersections.drop_duplicates(
    subset=["ID_Perfil", "ID_LineaCosta"],
    keep="first",
).copy()

# Shapely's project() returns the along-profile distance in the CRS units (meters here).
profile_geometries = profiles.set_index("ID_Perfil").geometry
profiles_shoreline_intersections["shoreline_position_m"] = (
    profiles_shoreline_intersections.apply(
        lambda row: profile_geometries.loc[row["ID_Perfil"]].project(row.geometry),
        axis=1,
    )
)

profiles_shoreline_intersections = gpd.GeoDataFrame(
    profiles_shoreline_intersections[
        ["ID_Perfil", "ID_playa"] + columns_to_keep + ["shoreline_position_m", "geometry"]
    ],
    geometry="geometry",
    crs=profiles.crs,
)
# Normalize shoreline_position_m per profile relative to the first available date.
# For each profile, subtract the position recorded at the earliest datetime
# so that the time series of every profile starts at 0.
_first_pos = (
    profiles_shoreline_intersections
    .sort_values(["ID_Perfil", "Fecha", "Hora"])
    .groupby("ID_Perfil")["shoreline_position_m"]
    .first()
    .rename("_first_pos")
)
profiles_shoreline_intersections = profiles_shoreline_intersections.merge(
    _first_pos, how="left", on="ID_Perfil"
)
profiles_shoreline_intersections["shoreline_position_m"] = (
    profiles_shoreline_intersections["shoreline_position_m"]
    - profiles_shoreline_intersections["_first_pos"]
)
profiles_shoreline_intersections.drop(columns=["_first_pos"], inplace=True)

# Create a single column for the date and time of the shoreline measurement
profiles_shoreline_intersections["datetime"] = pd.to_datetime(
    profiles_shoreline_intersections["Fecha"].astype(str)
    + " "
    + profiles_shoreline_intersections["Hora"].astype(str),
    #format="%Y%m%d %H%M",
    errors="coerce",
)
# Add the same column to the shorelines GeoDataFrame
shorelines["datetime"] = pd.to_datetime(
    shorelines["Fecha"].astype(str) + " " + shorelines["Hora"].astype(str),
    #format="%Y%m%d %H%M",
    errors="coerce",
)

In [4]:
# Remove Landsat images
profiles_shoreline_intersections = profiles_shoreline_intersections[
    profiles_shoreline_intersections["Mision"] == "S21C"
]

season_map = {
    1: 'Invierno', 2: 'Invierno', 3: 'Primavera',
    4: 'Primavera', 5: 'Primavera', 6: 'Verano',
    7: 'Verano', 8: 'Verano', 9: 'Otoño',
    10: 'Otoño', 11: 'Otoño', 12: 'Invierno'
}
profiles_shoreline_intersections['Estacion'] = profiles_shoreline_intersections['datetime'].dt.month.map(season_map)

# Keep only Verano and Primavera seasons
profiles_shoreline_intersections = profiles_shoreline_intersections[
    profiles_shoreline_intersections['Estacion'].isin(['Primavera', 'Verano', 'Otoño'])
]

In [5]:
from utils import SmartTidalFilter

# Initialize the filter with all available parameters explicitly defined
tidal_filter = SmartTidalFilter(
    min_years_span=5.0,                     # Minimum temporal span required for a tidal window (in years)
    max_gap_years=1.0,                      # Maximum continuous gap allowed between consecutive data points (in years)
    lambda_1=2.0,                           # Penalty weight for the amplitude of the tidal window (reduces topographic noise)
    lambda_2=1.5,                           # Penalty weight for losing recent data (penalizes windows that drop present years)
    window_sizes=[0.2, 0.4, 0.6, 0.8, 1.0], # List of tidal window widths to test during the optimization search (in meters)
    step_size=0.1,                          # Step size to slide the tidal window across the beach's tidal range (in meters)
    beach_col="ID_playa",                   # DataFrame column name containing the beach or spatial unit identifier
    tide_col="NivelTotal",                  # DataFrame column name containing the sea level/tidal height values
    date_col="datetime"                     # DataFrame column name containing the pre-processed datetime objects
)

# Apply the filter to the geospatial dataset
filtered_data = tidal_filter.fit_transform(profiles_shoreline_intersections)

In [6]:
from utils import ShorelineEvolutionAnalyzer

# Initialize the analyzer with a N-day temporal window
analyzer = ShorelineEvolutionAnalyzer(
    window='360D',               # Time-based window
    min_periods=2,               # Needs at least 2 points in those N days to compute a median
    profile_col="ID_Perfil",
    date_col="datetime",         # Must match the datetime column outputted by the tidal filter
    pos_col="shoreline_position_m"
)

# 1. Ejecutar el pipeline completo sobre los datos previamente filtrados por marea
smoothed_data = analyzer.fit_transform(filtered_data)

# 2. Extraer la tabla final con las tasas de evolución (m/año) y estadísticos
tasas_resultados = analyzer.rates_

# Ver los primeros resultados
display(tasas_resultados.head())

,ID_Perfil,n_observations,rate_ols_m_yr,r_squared,p_value,rate_theilsen_m_yr,theilsen_ci_lower,theilsen_ci_upper,is_significant
0,1,78,9.882,0.964,0.0,9.823,9.258,10.209,True
1,2,78,9.166,0.977,0.0,9.260,8.857,9.580,True
2,3,78,8.319,0.966,0.0,8.435,8.110,8.767,True
3,4,78,7.346,0.948,0.0,7.292,7.025,7.718,True
4,5,78,6.374,0.916,0.0,6.499,6.029,7.034,True


In [7]:
# Add a binary field indicating whether the intersection is filtered
smoothed_data['is_filtered'] = 1

# Join relevant attributes from smoothed_data to profiles_shoreline_intersections
profiles_shoreline_intersections = (
    profiles_shoreline_intersections.merge(
        smoothed_data[['ID_LineaCosta', 'ID_Perfil', 'is_filtered', 'shoreline_position_m_smoothed']],
        how='left',
        on=['ID_LineaCosta', 'ID_Perfil']
        )
)
profiles_shoreline_intersections['is_filtered'] = (
    profiles_shoreline_intersections['is_filtered'].fillna(0)
    .astype(int)
)

In [8]:
# Keep one authoritative beach per profile and include profiles without a rate.
perfil_playa = profiles[["ID_Perfil", "ID_playa"]].drop_duplicates(
    subset=["ID_Perfil"],
    keep="first",
)
tasas_resultados = tasas_resultados.merge(
    perfil_playa,
    how="right",
    on="ID_Perfil",
    validate="many_to_one",
)
assert tasas_resultados["ID_Perfil"].is_unique

In [9]:
# Identificar playas sin ninguna tasa calculable tras el filtrado.
perfil_playa = profiles[["ID_Perfil", "ID_playa", "provincia"]].drop_duplicates(
    subset=["ID_Perfil"], keep="first"
)

analisis_perfiles = perfil_playa.merge(
    tasas_resultados[["ID_Perfil", "rate_theilsen_m_yr"]],
    how="left",
    on="ID_Perfil",
    validate="one_to_one",
)
analisis_perfiles["tasa_calculable"] = analisis_perfiles["rate_theilsen_m_yr"].notna()
playas_con_tasa = (
    analisis_perfiles.loc[analisis_perfiles["tasa_calculable"], "ID_playa"]
    .drop_duplicates()
)
playas_sin_tasa = (
    analisis_perfiles.loc[
        ~analisis_perfiles["ID_playa"].isin(playas_con_tasa),
        ["ID_playa", "provincia"],
    ]
    .drop_duplicates(subset=["ID_playa"], keep="first")
)

# Calcular la longitud por relación espacial, no por coincidencia de nombres.
# Cada ID_LineaCosta se asigna a la Playa autoritativa de los perfiles que intersecta.
# lineas_por_playa = (
#     profiles_shoreline_intersections[["ID_LineaCosta", "ID_playa"]]
#     .dropna(subset=["ID_playa"])
#     .drop_duplicates()
# )
# longitudes_lineas = (
#     shorelines[["ID_LineaCosta", "geometry"]]
#     .drop_duplicates(subset=["ID_LineaCosta"], keep="first")
#     .assign(longitud_linea_m=lambda df: df.geometry.length)
#     [["ID_LineaCosta", "longitud_linea_m"]]
# )
# longitud_mediana_playa = (
#     lineas_por_playa
#     .merge(
#         longitudes_lineas,
#         how="inner",
#         on="ID_LineaCosta",
#         validate="many_to_one",
#     )
#     .groupby("ID_playa", as_index=False)["longitud_linea_m"]
#     .median()
#     .rename(columns={"longitud_linea_m": "longitud_mediana_m"})
# )

# playas_sin_tasa = playas_sin_tasa.merge(
#     longitud_mediana_playa,
#     how="left",
#     on="ID_playa",
#     validate="many_to_one",
# )
# playas_sin_tasa = playas_sin_tasa[
#     ["ID_playa", "longitud_mediana_m", "provincia"]
# ].sort_values("ID_playa").reset_index(drop=True)

# playas_sin_tasa['is_short'] = playas_sin_tasa['longitud_mediana_m'] < 200

print(f"Número de playas sin tasa calculable: {len(playas_sin_tasa)}")
display(playas_sin_tasa)

# print(f"Número de playas cortas sin tasa calculable: {playas_sin_tasa['is_short'].sum()}")

# Exportación opcional:
playas_sin_tasa.to_csv("playas_sin_tasa_min_year_1_summ_spr_fall.csv", index=False, encoding="utf-8-sig")

Número de playas sin tasa calculable: 75


,ID_playa,provincia
14353,3365,Murcia
14375,3369,Murcia
15180,1918,Alacant/Alicante
15868,1908,Alacant/Alicante
16405,1938,Alacant/Alicante
...,...,...
37565,3294,Illes Balears
37569,3010,Illes Balears
38159,3071,Illes Balears
38203,3161,Illes Balears


In [10]:
# Add a binary column in shorelines dataset indicating whether the shoreline is filtered or not
# A shoreline can intersect several profiles, so reduce it to one flag per
# ID_LineaCosta before merging.
filtered_by_line = (
    profiles_shoreline_intersections
    .groupby("ID_LineaCosta", as_index=False)["is_filtered"]
    .max()
)

# Avoid duplicate-column errors if this cell is executed again.
shorelines = shorelines.drop(columns=["is_filtered"], errors="ignore")

shorelines = shorelines.merge(
    filtered_by_line,
    how="left",
    on="ID_LineaCosta",
    validate="one_to_one",
)

# Shorelines not intersecting a profile are not filtered.
shorelines["is_filtered"] = (
    shorelines["is_filtered"]
    .fillna(0)
    .astype(int)
)

In [11]:
# Final post-processing to ensure only one representative shoreline
# exists per Playa and year.

# Restore all representative shorelines to the regular filtered value.
# This removes any previous value 2 before recalculating it.
shorelines.loc[
    shorelines["is_filtered"].eq(2),
    "is_filtered",
] = 1

# Temporary columns for the final deterministic selection.
shorelines["_year"] = shorelines["datetime"].dt.year
shorelines["_target_date"] = pd.to_datetime(
    shorelines["_year"].astype(str) + "-08-15"
)
shorelines["_time_to_target"] = (
    shorelines["datetime"] - shorelines["_target_date"]
).abs()
shorelines["_row_order"] = range(len(shorelines))

final_candidates = (
    shorelines[
        shorelines["is_filtered"].eq(1)
        & shorelines["Playa"].notna()
        & shorelines["_year"].notna()
        & shorelines["_time_to_target"].notna()
    ]
    .sort_values(
        [
            "Playa",
            "_year",
            "_time_to_target",
            "_row_order",
        ],
        kind="stable",
    )
    .groupby(
        ["Playa", "_year"],
        sort=False,
    )
    .head(1)
)

# Mark only the selected row in each Playa-year group as representative.
shorelines.loc[
    shorelines["_row_order"].isin(final_candidates["_row_order"]),
    "is_filtered",
] = 2

# Optional validation: there must be at most one value 2 per Playa and year.
duplicate_representatives = (
    shorelines.loc[
        shorelines["is_filtered"].eq(2)
        & shorelines["Playa"].notna(),
    ]
    .groupby(
        [
            "Playa",
            shorelines.loc[
                shorelines["is_filtered"].eq(2)
                & shorelines["Playa"].notna(),
                "_year",
            ],
        ],
    )
    .size()
)

assert duplicate_representatives.le(1).all(), (
    "There is more than one representative shoreline "
    "for at least one Playa-year group."
)

# Remove temporary columns.
shorelines.drop(
    columns=[
        "_year",
        "_target_date",
        "_time_to_target",
        "_row_order",
    ],
    inplace=True,
)

# Export the shorelines dataset.
shorelines.to_file(
    os.path.join(
        "data",
        "shorelines",
        "LineasCosta_with_filter_flags.gpkg",
    ),
    layer="LineasCostaPlayas",
    driver="GPKG",
)

In [12]:
# Add the empty_playa_rows back to the profiles GeoDataFrame to ensure all profiles are represented.
profiles = pd.concat([profiles, empty_playa_rows], ignore_index=True)

# Export the profiles dataset
profiles.to_file(
    os.path.join("data", "profiles", "perfiles_limpios.gpkg"),
    layer="perfiles_limpios",
    driver="GPKG"
)

In [13]:
# Rename some columns and keep only the relevant ones
tasas_resultados = tasas_resultados[
    ['ID_playa', 'ID_Perfil', 'n_observations',
     'rate_theilsen_m_yr', 'theilsen_ci_lower', 'theilsen_ci_upper',
     'is_significant'
     ]
    ]
tasas_resultados = tasas_resultados.rename(
    columns={
        'rate_theilsen_m_yr': 'rate_m_yr',
        'theilsen_ci_lower': 'rate_ci_lower',
        'theilsen_ci_upper': 'rate_ci_upper'
    }
)

# Export the DataFrame to a CSV file
tasas_resultados.to_csv(
    'tasas_perfiles.csv', index=False
)

In [14]:
# Rename some columns and keep only the relevant ones
intersections_to_export = profiles_shoreline_intersections.loc[
    profiles_shoreline_intersections['is_filtered'] == 1
    ].copy()

intersections_to_export = intersections_to_export[
   [ 'ID_Perfil', 'datetime', 'shoreline_position_m', 'shoreline_position_m_smoothed', 'NivelTotal']
   ]

intersections_to_export = intersections_to_export.rename(
    columns={
        'datetime': 'Fecha',
        'shoreline_position_m': 'Distancia',
        'shoreline_position_m_smoothed': 'Distancia_Suavizada',
        'NivelTotal': 'Nivel_Total'
    }
)
intersections_to_export.drop(columns=['Distancia']).to_csv(
    'S2_seatool.csv', index=False
)
intersections_to_export.to_csv(
    'S2_seatool_full.csv', index=False
)

In [15]:
# # Visualización conjunta de perfiles seleccionados y sus tendencias temporales.
# import contextily as ctx
# from scipy.stats import theilslopes

# profile_min, profile_max = 27474, 27484
# profiles_window = profiles.loc[profiles["ID_Perfil"].between(profile_min, profile_max)].copy()
# if profiles_window.empty:
#     raise ValueError(f"No hay perfiles con ID_Perfil entre {profile_min} y {profile_max}.")

# available_ids = sorted(profiles_window["ID_Perfil"].unique())
# selected_ids = [available_ids[0], available_ids[len(available_ids) // 2], available_ids[-1]]
# selected_ids = sorted(
#     selected_ids,
#     key=lambda profile_id: profiles_window.loc[
#         profiles_window["ID_Perfil"].eq(profile_id), "geometry"
#     ].iloc[0].centroid.y,
# )
# if len(selected_ids) != 3:
#     raise ValueError("El intervalo debe contener al menos tres perfiles.")

# rates = tasas_resultados.rename(columns={
#     "rate_m_yr": "rate_theilsen_m_yr",
#     "rate_ci_lower": "theilsen_ci_lower",
#     "rate_ci_upper": "theilsen_ci_upper",
# }).copy()
# required = {"ID_Perfil", "rate_theilsen_m_yr", "theilsen_ci_lower", "theilsen_ci_upper"}
# missing = required.difference(rates.columns)
# if missing:
#     raise KeyError("Faltan columnas de tasas: " + ", ".join(sorted(missing)))

# observations = profiles_shoreline_intersections.loc[
#     profiles_shoreline_intersections["ID_Perfil"].isin(selected_ids)
#     & profiles_shoreline_intersections["datetime"].notna()
#     & profiles_shoreline_intersections["shoreline_position_m"].notna()
# ].copy()
# if observations.empty:
#     raise ValueError("No hay observaciones temporales para los perfiles seleccionados.")
# observations = (observations.sort_values(["ID_Perfil", "datetime"])
#                 .drop_duplicates(["ID_Perfil", "datetime"], keep="first"))

# colors = dict(zip(selected_ids, ["#d62728", "#2ca02c", "#1f77b4"]))
# fig = plt.figure(figsize=(16, 10), constrained_layout=True)
# grid = fig.add_gridspec(nrows=3, ncols=2, width_ratios=[1.05, 1.65])
# map_ax = fig.add_subplot(grid[:, 0])
# profiles_window.plot(ax=map_ax, color="lightgray", linewidth=1.0, alpha=0.8)
# for profile_id in selected_ids:
#     profiles_window.loc[profiles_window["ID_Perfil"].eq(profile_id)].plot(
#         ax=map_ax, color=colors[profile_id], linewidth=2.8)
# for profile_id in selected_ids:
#     geometry = profiles_window.loc[profiles_window["ID_Perfil"].eq(profile_id), "geometry"].iloc[0]
#     point = geometry.interpolate(0.5, normalized=True)
#     map_ax.annotate(str(profile_id), (point.x, point.y), xytext=(5, 5), textcoords="offset points",
#                     color=colors[profile_id], fontsize=9, fontweight="bold")
# ctx.add_basemap(
#     map_ax,
#     source=ctx.providers.Esri.WorldImagery,
#     crs=profiles_window.crs,
#     alpha=0.5,
# )
# map_ax.set_title(f"Perfiles {profile_min}-{profile_max}")
# map_ax.set_axis_off()

# for row, profile_id in enumerate(selected_ids[::-1]):
#     ax = fig.add_subplot(grid[row, 1])
#     profile_data = observations.loc[observations["ID_Perfil"].eq(profile_id)].sort_values("datetime")
#     ax.plot(profile_data["datetime"], profile_data["shoreline_position_m"], "o",
#             color="gray", alpha=0.35, markersize=2.5, label="Todos los datos")
#     smooth_data = profile_data.loc[profile_data["shoreline_position_m_smoothed"].notna()].copy()
#     if len(smooth_data) < 3:
#         raise ValueError(f"El perfil {profile_id} no tiene al menos tres datos suavizados.")
#     x_years = ((smooth_data["datetime"] - pd.Timestamp("1970-01-01")).dt.total_seconds()
#                / (365.2425 * 24 * 60 * 60))
#     y_smoothed = smooth_data["shoreline_position_m_smoothed"].to_numpy()
#     sen_slope, sen_intercept, _, _ = theilslopes(y_smoothed, x_years, alpha=0.95)
#     ax.plot(smooth_data["datetime"], y_smoothed, "o", color=colors[profile_id],
#             markersize=5, label="Datos suavizados")
#     ax.plot(smooth_data["datetime"], sen_intercept + sen_slope * x_years, color="black",
#             linestyle="--", linewidth=1.5, label="Ajuste Theil-Sen")
#     rate_row = rates.loc[rates["ID_Perfil"].eq(profile_id)]
#     if rate_row.empty:
#         raise ValueError(f"No hay tasa Theil-Sen para el perfil {profile_id}.")
#     rate_row = rate_row.iloc[0]
#     ax.text(0.02, 0.05,
#             f"Theil-Sen: {rate_row['rate_theilsen_m_yr']:.3f} m/año\n"
#             f"CI 95%: [{rate_row['theilsen_ci_lower']:.3f}, "
#             f"{rate_row['theilsen_ci_upper']:.3f}] m/año",
#             transform=ax.transAxes, fontsize=9,
#             bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "0.7"})
#     ax.set_title(f"Perfil {profile_id}")
#     ax.set_ylabel("Posición de la línea de costa (m)")
#     ax.grid(alpha=0.25)
#     ax.legend(loc="best", fontsize=8)

# fig.axes[-1].set_xlabel("Fecha")
# plt.show()


In [16]:
profiles.columns

Index(['OBJECTID_1', 'id_prov', 'tipo', 'provincia', 'Playa', 'xon', 'yon',
       'xof', 'yof', 'X', 'Y', 'dist_tierr', 'dist_lnero', 'dist_lref',
       'Playa_norm', 'geom_Lengt', 'ID_Perfil', 'ID_playa', 'Shape_Leng',
       'geometry'],
      dtype='str')